# QB dropback ideas 

In this notebook I want to start my research into how offense formation and the nummber of down and distance fromm first down line/touchdown affects whether a QB dropsback or Handsoff.

I will do this by testing the data using the statistical tests, a linear regression model, and a XGBoost model

I will be working through the following tasks:

- Decide scope: whole league, or a subset.
- Build the feature set table above into an actual dataframe.
- Bucket yards_to_go into short/medium/long?.
- Run chi-squared tests on each feature vs. is_dropback; record p-value + effect size for each.
- Compute the baseline.
- Fit logistic regression; extract coefficients as odds ratios.
- Fit XGBoost on the same features; compare metrics against logistic regression and the baseline.
- Write pytest tests.
- Write up results: chi-squared findings, logistic regression coefficients/odds ratios, logistic-vs-XGBoost comparison, and a conclusion on how down/distance/formation actually affect dropback likelihood.


## Scope

In [1]:
#setup
import sys
sys.path.append('../src')

import pandas as pd

plays = pd.read_csv('/Users/seanlee/Desktop/Github_repositories/Practice-repo/data/plays.csv')

print(f"plays: {plays.shape[0]} rows, {plays['gameId'].nunique()} games, {plays['possessionTeam'].nunique()} teams")
print(plays['isDropback'].value_counts(normalize=True).rename('share'))

plays: 16124 rows, 136 games, 32 teams
isDropback
True     0.60382
False    0.39618
Name: share, dtype: float64


## Feature dataframe and bucket yards

Excluding qbKneel, qbSneak, and qbSpike plays. Also dropping the small number of rows with a missing offenseFormation, since it's one of our features.

Features: down, yardsToGo (raw + bucketed into short/medium/long), offenseFormation. Target: isDropback.

In [2]:
#build the feature dataframe

from dropback_analysis.features import build_features

features = build_features(plays)

print(f"features: {features.shape[0]} rows (dropped {plays.shape[0] - features.shape[0]} plays)")
features.head()

features: 15816 rows (dropped 308 plays)


,gameId,playId,down,yardsToGo,offenseFormation,isDropback,yardsToGoBucket
0,2022102302,2655,1,10,EMPTY,True,long
1,2022091809,3698,1,10,EMPTY,True,long
2,2022103004,3146,3,12,SHOTGUN,True,long
3,2022110610,348,2,10,SHOTGUN,True,long
4,2022102700,2799,2,8,PISTOL,False,long


## Run chi-quared test for features to check relevance

Record the p-value for down, yardsToGoBucket, and offenseFormation vs isDropback and check effect size of each.

In [3]:
#chi-square test of independence, generalized to any feature vs isDropback

import numpy as np
from dropback_analysis.stats import chi_square_test

chi2_results = {f: chi_square_test(features, f) for f in ['offenseFormation', 'down', 'yardsToGoBucket']}

offenseFormation vs isDropback
p-value: 0.00e+00
cramer's v: 0.459
significant (p < 0.05) - offenseFormation and isDropback are not independent

chi2 contribution by offenseFormation:
offenseFormation
SINGLEBACK    36.6%
EMPTY         21.0%
SHOTGUN       19.2%
I_FORM        13.0%
PISTOL         5.6%
JUMBO          2.5%
WILDCAT        2.0%
dtype: str

down vs isDropback
p-value: 1.51e-206
cramer's v: 0.246
significant (p < 0.05) - down and isDropback are not independent

chi2 contribution by down:
down
3    59.0%
1    37.3%
4     3.3%
2     0.4%
dtype: str

yardsToGoBucket vs isDropback
p-value: 1.55e-42
cramer's v: 0.110
significant (p < 0.05) - yardsToGoBucket and isDropback are not independent

chi2 contribution by yardsToGoBucket:
yardsToGoBucket
short     72.3%
medium    26.7%
long       1.0%
dtype: str



## Baseline

Before fitting any model, set the bar it actually needs to clear: if you always predicted the majority class with no features at all, how accurate would you be? Any model that can't beat this isn't adding value.

In [4]:
#majority-class baseline

from dropback_analysis.models import majority_class_baseline

baseline = majority_class_baseline(features)

print(baseline['class_shares'].rename('share'))
print(f"\nmajority class: {baseline['majority_class']}")
print(f"baseline accuracy (always predict {baseline['majority_class']}): {baseline['baseline_accuracy']:.3f}")

isDropback
True     0.614125
False    0.385875
Name: share, dtype: float64

majority class: True
baseline accuracy (always predict True): 0.614


# Logistic regression



Fitting isDropback on offenseFormation, down, and yardsToGoBucket. All three are treated as categories since the chi-squared step showed their effect isn't linear.

In [5]:
#fit logistic regression

from dropback_analysis.models import fit_logistic_regression

logit_model = fit_logistic_regression(features)

print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.532276
         Iterations 7
                             Logit Regression Results                             
Dep. Variable:     isDropback.astype(int)   No. Observations:                15816
Model:                              Logit   Df Residuals:                    15804
Method:                               MLE   Df Model:                           11
Date:                    Thu, 27 Aug 2026   Pseudo R-squ.:                  0.2018
Time:                            15:05:50   Log-Likelihood:                -8418.5
converged:                           True   LL-Null:                       -10547.
Covariance Type:                nonrobust   LLR p-value:                     0.000
                                                                        coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------

In [6]:
#same predict() output, but lined up against the actual play so the link to the coefficients is obvious

predicted_probability = logit_model.predict()

preview = features[['offenseFormation', 'down', 'yardsToGoBucket', 'isDropback']].copy()
preview['predicted_dropback_probability'] = predicted_probability
preview.head()

,offenseFormation,down,yardsToGoBucket,isDropback,predicted_dropback_probability
0,EMPTY,1,long,True,0.948054
1,EMPTY,1,long,True,0.948054
2,SHOTGUN,3,long,True,0.907610
3,SHOTGUN,2,long,True,0.798412
4,PISTOL,2,long,False,0.472990


In [7]:
#every unique formation/down/yardsToGoBucket combination that occurs in the data, with its predicted probability
#(there are only up to 7*4*3=84 possible combos, so many of the 15,816 plays share the exact same prediction)

unique_combos = (
    features.groupby(['offenseFormation', 'down', 'yardsToGoBucket'], observed=True)
    .size()
    .rename('n_plays')
    .reset_index()
)
unique_combos['predicted_dropback_probability'] = logit_model.predict(unique_combos)

print(f"{len(unique_combos)} unique combinations occur in the data")
unique_combos.sort_values('predicted_dropback_probability', ascending=False)

74 unique combinations occur in the data


,offenseFormation,down,yardsToGoBucket,n_plays,predicted_dropback_probability
11,EMPTY,4,long,12,0.993327
10,EMPTY,4,medium,7,0.989823
8,EMPTY,3,long,240,0.988254
7,EMPTY,3,medium,180,0.982134
9,EMPTY,4,short,17,0.973899
...,...,...,...,...,...
53,SINGLEBACK,1,short,41,0.106951
12,I_FORM,1,short,15,0.089218
67,WILDCAT,2,short,8,0.078670
22,JUMBO,1,short,22,0.064421


# XGBoost

Split into train/test, and re-evaluate the baseline and logistic regression on the same held-out test set as XGBoost, using the same one-hot encoded features for both models so it's a fair comparison.

In [8]:
#train/test split, one-hot encode the same three features for both models

from sklearn.model_selection import train_test_split

X = pd.get_dummies(features[['offenseFormation', 'down', 'yardsToGoBucket']], columns=['offenseFormation', 'down', 'yardsToGoBucket'], drop_first=True)
y = features['isDropback'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"train: {X_train.shape[0]} rows, test: {X_test.shape[0]} rows")
print(f"train dropback rate: {y_train.mean():.3f}, test dropback rate: {y_test.mean():.3f}")

train: 12652 rows, test: 3164 rows
train dropback rate: 0.614, test dropback rate: 0.614


In [9]:
#baseline and logistic regression, evaluated on the held-out test set

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

baseline_test_pred = pd.Series(y_train.mode()[0], index=y_test.index)
baseline_test_accuracy = accuracy_score(y_test, baseline_test_pred)

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
logreg_pred = logreg.predict(X_test)
logreg_proba = logreg.predict_proba(X_test)[:, 1]

print(f"baseline test accuracy: {baseline_test_accuracy:.3f}")
print(f"logistic regression test accuracy: {accuracy_score(y_test, logreg_pred):.3f}")

baseline test accuracy: 0.614
logistic regression test accuracy: 0.748


In [10]:
#fit xgboost on the same train/test split and features

from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print(f"xgboost test accuracy: {accuracy_score(y_test, xgb_pred):.3f}")

xgboost test accuracy: 0.745


In [11]:
#compare all three on the same held-out test set

comparison = pd.DataFrame([
    {'model': 'baseline', 'accuracy': baseline_test_accuracy, 'log_loss': np.nan, 'roc_auc': np.nan},
    {'model': 'logistic regression', 'accuracy': accuracy_score(y_test, logreg_pred),
     'log_loss': log_loss(y_test, logreg_proba), 'roc_auc': roc_auc_score(y_test, logreg_proba)},
    {'model': 'xgboost', 'accuracy': accuracy_score(y_test, xgb_pred),
     'log_loss': log_loss(y_test, xgb_proba), 'roc_auc': roc_auc_score(y_test, xgb_proba)},
])
comparison

,model,accuracy,log_loss,roc_auc
0,baseline,0.614096,NaN,NaN
1,logistic regression,0.748104,0.521705,0.802251
2,xgboost,0.745259,0.518540,0.802586


# Summary of findings

## Chi-squared: does each feature actually matter?

All three features are statistically significant (p < 0.05), but they are not equally important — Cramér's V shows a clear ranking:

| feature | Cramér's V | strength | biggest driver |
|---|---|---|---|
| offenseFormation | 0.459 | strongest | SINGLEBACK (36.6% of chi2), EMPTY (21.0%) |
| down | 0.246 | moderate | 3rd down (59.0% of chi2) |
| yardsToGoBucket | 0.110 | weak | short yardage (72.3% of chi2) |

**Formation is by far the dominant signal.** Down matters, but mostly through the 3rd-down effect specifically. Distance-to-go matters least of the three, and what little effect it has is concentrated in short-yardage situations.

## Logistic regression: odds ratios

(All relative to the reference play: SHOTGUN, 1st down, short yardage.)

In [12]:
#clean odds ratio table for the write-up

from dropback_analysis.models import odds_ratio_table

odds_ratios = odds_ratio_table(logit_model)
odds_ratios

,coef,odds_ratio,p_value,significant
offenseFormation: EMPTY,2.147613,8.564392,1.080018e-44,True
down: 4,2.098803,8.156398,1.168440e-33,True
down: 3,1.528196,4.609851,1.848533e-102,True
yardsToGoBucket: long,1.383701,3.989639,1.397337e-91,True
yardsToGoBucket: medium,0.958130,2.606817,8.602637e-42,True
down: 2,0.619801,1.858559,7.795841e-37,True
Intercept,-0.627100,0.534138,7.393132e-17,True
offenseFormation: PISTOL,-1.484545,0.226605,8.291197e-64,True
offenseFormation: SINGLEBACK,-1.495175,0.224209,6.145807e-258,True
offenseFormation: I_FORM,-1.696114,0.183395,2.838662e-114,True


- **Formation swings the odds the most, in both directions.** EMPTY formation is 8.6x more likely to be a dropback than SHOTGUN. 
- WILDCAT, JUMBO, I_FORM, SINGLEBACK, and PISTOL are all less likely than SHOTGUN (odds ratios 0.09–0.23) — these are the run-oriented personnel groupings.
- **Down effects escalate**: 2nd down 1.9x, 3rd down 4.6x, 4th down 8.2x more likely to dropback than 1st down. Later downs push teams toward passing to convert.
- **Distance effects also escalate**: medium yardage 2.6x, long yardage 4.0x more likely to dropback than short yardage.
- Every one of these effects is statistically significant (p < 0.05).

## Logistic regression vs. XGBoost

Evaluated on the same held-out test set (3,164 plays), with the same one-hot encoded features:

| model | accuracy | log loss | ROC-AUC |
|---|---|---|---|
| baseline (majority class) | 0.614 | — | — |
| logistic regression | 0.748 | 0.522 | 0.802 |
| xgboost | 0.745 | 0.519 | 0.803 |

Both models beat the baseline by 13 points of accuracy formation/down/distance all have predictive signal. But logistic regression and XGBoost are statistically tied. Since XGBoost's main advantage over logistic regression is picking up non-linear effects and feature interactions (e.g. "3rd down and long yardage and SHOTGUN" isn't behaving differently than what you'd predict by combining each effect independently).

## Conclusion

Offense formation, down, and distance-to-go all genuinely affect whether a QB drops back or the offense hands off — none of this is noise. But they don't matter equally:

1. **Formation is the primary driver.** It's not just the strongest statistical association (Cramér's V 0.459 vs. 0.246 and 0.110) — it also produces the single largest odds swing in the model (EMPTY at ~8.6x SHOTGUN). This makes sense: formation is close to a proxy for personnel and play design intent (an empty backfield removes the option to hand off almost entirely).
2. **Down and distance both matter, and in the expected direction**, later downs and longer distances both push toward passing, but their effects are smaller than formation's, and distance is the weakest of the three.
3. **A model using only these three simple, pre-snap-observable features reaches 75% accuracy / 0.80 AUC**, a solid improvement over the 61% baseline. XGBoost matching (not beating) logistic regression suggests these three features' effects are essentially additive — there isn't a hidden interaction effect being missed by the simpler model.